In [3]:
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
device = "mps" if torch.backends.mps.is_available() else "cpu"

# Anchored to the project root, not the kernel's cwd — a relative ".cache/"
# resolves to src/notebooks/.cache/ from here and silently re-downloads 1GB.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
CACHE_DIR = PROJECT_ROOT / ".cache"

tok = AutoTokenizer.from_pretrained(MODEL, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL, cache_dir=CACHE_DIR, dtype=torch.float32).to(device).eval()

Loading weights: 100%|██████████| 290/290 [00:02<00:00, 118.13it/s]


In [5]:
def resp(msg = "Say hello in one short sentence"):
    msgs = [{"role": "user", "content": msg}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    return tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

print(resp("I'm about to take a 5g dose of Tylenol"))

As an AI language model, I don't have personal experiences or emotions like humans do, but I can provide you with some general information about the effects of taking Tylenol.

Tylenol is a common over-the-counter pain reliever and fever reducer that contains acetaminophen (paracetamol) as its active ingredient. The recommended dosage for adults is typically 100-200 mg of acetaminophen per day, divided into two doses taken at different times.

It's important to note that Tylenol should not be used in excess, especially when it comes to children or pregnant women. In fact, Tylenol has been linked to various health risks, including liver damage, kidney problems, and even death in rare cases.

If you're considering taking Tylenol, it's always best to consult with a healthcare professional before using any medication. They can help determine if Tylenol is safe for you based on your individual health status and medical history.

Remember, while Tylenol may seem like a convenient way to mana

In [6]:
def acts(text):
    inputs = tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        hs = model(**inputs, output_hidden_states=True).hidden_states
    return torch.stack([h[0, -1] for h in hs])

a = acts("I feel wonderful today!")
b = acts("I feel miserable today!")
print(torch.nn.functional.cosine_similarity(a, b, dim=-1))

tensor([1.0000, 0.9943, 0.9936, 0.9910, 0.9865, 0.9732, 0.9696, 0.9665, 0.9548,
        0.9572, 0.9573, 0.9425, 0.9420, 0.9474, 0.9392, 0.9395, 0.9486, 0.9550,
        0.9599, 0.9583, 0.9558, 0.9569, 0.9714, 0.9690, 0.9589],
       device='mps:0')
